In [ ]:
%pip install langchain-openai python-dotenv "pydantic>=2,<3"

In [1]:
from pydantic import BaseModel, Field

class OrderItem(BaseModel):
    product: str
    quantity: int = Field(ge = 1, le = 20, strict = True, description = "Quantity must be between 1 and 20")

In [4]:
item = OrderItem(product="cold_coffee", quantity=5)  # Valid
print(item.product)
print(item.quantity)

cold_coffee
5


In [5]:
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()
llm = ChatOpenAI(
    model="google/gemini-2.5-flash",
    api_key=os.environ["OPEN_ROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
)

In [10]:
## How do I tell LLM that you need to give me output in this structure?
item_llm = llm.with_structured_output(OrderItem)
item = item_llm.invoke("I want to order 5 cold coffees")
print(item.product)
print(item.quantity)

cold coffees
5


/Users/sameerdubey/miniconda3/envs/ai-dev/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=OrderItem(product='cold coffees', quantity=5), input_type=OrderItem])
  return self.__pydantic_serializer__.to_python(


In [8]:
llm.invoke("I want to order 5 cold coffees")

AIMessage(content='Sure, I can help you with that. What kind of cold coffees would you like? Do you have any specific preferences, like:\n\n*   **Type of coffee:** Iced latte, iced americano, cold brew, frappuccino, etc.?\n*   **Size:** Small, medium, large?\n*   **Milk preference:** Dairy (whole, skim, etc.) or non-dairy (almond, oat, soy)?\n*   **Sweetener:** Sugar, syrup (vanilla, caramel), artificial sweetener, or unsweetened?\n*   **Any other additions:** Whipped cream, extra shot, flavorings?\n\nOnce I have these details, I can place the order for you!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 146, 'prompt_tokens': 8, 'total_tokens': 154, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'video_tokens': 0}, 'cost': 0.0003674, '

In [9]:
item_llm.invoke("Hi How are you?")

/Users/sameerdubey/miniconda3/envs/ai-dev/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=OrderItem(product='unknown', quantity=1), input_type=OrderItem])
  return self.__pydantic_serializer__.to_python(


OrderItem(product='unknown', quantity=1)

In [11]:
from pydantic import ValidationError

try:
    OrderItem(product="cold coffee", quantity=-2)
except ValidationError as error:
    print(error)

1 validation error for OrderItem
quantity
  Input should be greater than or equal to 1 [type=greater_than_equal, input_value=-2, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/greater_than_equal


In [ ]:
from typing import Literal

class MenuItem(BaseModel):
    product_id: Literal[
        "cold_coffee", "paneer_tikka", "veg_sandwich", "lemonade"
    ]
    quantity: int = Field(ge=1, le=20, strict=True)

    #{
    }

In [13]:
menu_llm = llm.with_structured_output(MenuItem, method="function_calling")

item = menu_llm.invoke(
    "Extract the menu item and quantity from the following order:"
    "Customer: Two cold coffees, please and one paneer tikka please"
)
print(item.model_dump_json(indent=2))

{
  "product_id": "cold_coffee",
  "quantity": 2
}


In [14]:
class OrderSelection(BaseModel):
    items: list[MenuItem] = Field(min_length=1, max_length=4)

In [16]:
order_llm = llm.with_structured_output(
    OrderSelection
)

instructions = """Extract the customer's order using the menu product IDs.
Menu: Cold Coffee, Paneer Tikka, Veg Sandwich, Lemonade.
Use the quantities explicitly requested. Do not include prices or payment details.
"""

order = order_llm.invoke([
    ("system", instructions),
    ("human", "Two cold coffees and one paneer tikka, please."),
])
print(order.model_dump_json(indent=2))

{
  "items": [
    {
      "product_id": "cold_coffee",
      "quantity": 2
    },
    {
      "product_id": "paneer_tikka",
      "quantity": 1
    }
  ]
}


/Users/sameerdubey/miniconda3/envs/ai-dev/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=OrderSelection(items=[Men...er_tikka', quantity=1)]), input_type=OrderSelection])
  return self.__pydantic_serializer__.to_python(


In [21]:
from langchain_core.tools import tool

@tool
def loose_order(items: list[dict]):
    """Receive the customer's product IDs and quantities."""
    print("Loose tool function executed")
    return items



In [ ]:
bad_items = {"items": [{"product_id": "pizza", "quantity": -2}]}
loose_order.invoke(bad_items)

In [19]:
@tool(args_schema=OrderSelection)
def validated_order(items: list[MenuItem]):
    """Receive menu product IDs with integer quantities from 1 to 20."""
    print("Validated tool function executed")
    return [item.model_dump(mode="json") for item in items]



In [20]:
bad_items = {"items": [{"product_id": "pizza", "quantity": -2}]}
try:
    validated_order.invoke(bad_items)
except ValidationError as error:
    print(error)

2 validation errors for OrderSelection
items.0.product_id
  Input should be 'cold_coffee', 'paneer_tikka', 'veg_sandwich' or 'lemonade' [type=literal_error, input_value='pizza', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/literal_error
items.0.quantity
  Input should be greater than or equal to 1 [type=greater_than_equal, input_value=-2, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/greater_than_equal


In [22]:
from langchain_core.tools import tool

# Friendly names come from our menu, not from the model inventing a label.
product_names = {
    "cold_coffee": "Cold Coffee",
    "paneer_tikka": "Paneer Tikka",
    "veg_sandwich": "Veg Sandwich",
    "lemonade": "Lemonade",
}

# One demo order for this notebook; resets when the kernel restarts.
demo_order = {"items": []}

@tool(args_schema=OrderSelection)
def set_order(items: list[MenuItem]):
    """Save the customer's complete selection using menu product IDs and quantities."""
    demo_order["items"] = [
        {"product_id": item.product_id,
         "product_name": product_names[item.product_id],
         "quantity": item.quantity}
        for item in items
    ]
    return {"status": "draft_saved", "items": demo_order["items"]}

llm_with_order_tool = llm.bind_tools([set_order])


In [24]:
response = llm_with_order_tool.invoke(
    "Save my order using the menu product IDs: "
    "two cold coffees and one paneer tikka."
)
response

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'tool_set_order_9j4pKMTtZyizmEvzUcXM', 'function': {'arguments': '{"items":[{"quantity":2,"product_id":"cold_coffee"},{"product_id":"paneer_tikka","quantity":1}]}', 'name': 'set_order'}, 'type': 'function', 'index': 0}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 36, 'total_tokens': 91, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'video_tokens': 0}, 'cost': 0.0001483, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.0001483, 'upstream_inference_prompt_cost': 1.08e-05, 'upstream_inference_completions_cost': 0.0001375}}, 'model_name': 'google/gemini-2.5-flash', 'system_fingerprint': None, 'id': 'gen-1792083448-KGVNTmIFsz61B6JQITtQ', 'service_tier': 'defa

In [25]:
response.tool_calls

[{'name': 'set_order',
  'args': {'items': [{'quantity': 2, 'product_id': 'cold_coffee'},
    {'product_id': 'paneer_tikka', 'quantity': 1}]},
  'id': 'tool_set_order_9j4pKMTtZyizmEvzUcXM',
  'type': 'tool_call'}]

In [26]:
from langchain_core.tools import tool
from difflib import get_close_matches

class SearchMenuInput(BaseModel):
    query: str = Field(description="The product name to look for, as requested by the customer.")

products = [
    {"product_id": "P104", "name": "Cold Coffee"},
    {"product_id": "P208", "name": "Chocolate Cold Coffee"},
    {"product_id": "P301", "name": "Paneer Tikka"},
]

@tool(args_schema=SearchMenuInput)
def search_menu(query: str):
    """Find menu products matching a customer's product name, including minor typos."""
    names = [product["name"].lower() for product in products]
    matches = get_close_matches(query.lower(), names, n=3, cutoff=0.6)
    return [product for name in matches for product in products
            if product["name"].lower() == name]

In [27]:
# Force the search tool only to make this demonstration predictable.
llm_with_search = llm.bind_tools([search_menu], tool_choice="search_menu")
search_response = llm_with_search.invoke("Look for cold coffe on the menu.")
search_response.tool_calls

[{'name': 'search_menu',
  'args': {'query': 'cold coffee'},
  'id': 'tool_search_menu_Tx11OUtqebixbbOjBX6g',
  'type': 'tool_call'}]

In [28]:
# The model requested search. Our code actually executes it.
search_call = search_response.tool_calls[0]
search_result = search_menu.invoke(search_call)
print(search_result.content)

[{"product_id": "P104", "name": "Cold Coffee"}, {"product_id": "P208", "name": "Chocolate Cold Coffee"}]


In [29]:
from langchain_core.messages import HumanMessage

reply = llm.invoke([
    ("system", "Explain the menu matches using the tool result. If multiple products "
     "match, ask which one the customer wants. Do not place an order."),
    HumanMessage(content="Look for cold coffe on the menu."),
    search_response,
    search_result,
])
print(reply.content)

We have two types of cold coffee: Cold Coffee and Chocolate Cold Coffee. Which one would you like?


In [30]:
from langchain_core.messages import HumanMessage, SystemMessage

# Only the search tool is available in this example.
tools = [search_menu]
tool_repo = {tool.name: tool for tool in tools}
llm_with_tools = llm.bind_tools(tools)

system_message = SystemMessage(content="""You are a helpful kiosk assistant.
Use search_menu to find products before discussing matching items.
Use product names and IDs from the tool result; never invent them.
If multiple products match, ask which one the customer wants.
If no products match, explain that and ask for another product name.
Do not place an order. Give one clear answer after receiving the results.
Do not print IDs of products or anything
""")

user_query = input("What would you like to look for? ")
messages_to_send = [system_message, HumanMessage(content=user_query)]

response = llm_with_tools.invoke(messages_to_send)
while response.tool_calls:
    # Preserve the AI message containing the requested tools.
    messages_to_send.append(response)
    for tool_call in response.tool_calls:
        tool_name = tool_call["name"]
        # A full tool call produces a ToolMessage linked to that call.
        tool_output = tool_repo[tool_name].invoke(tool_call)
        messages_to_send.append(tool_output)
    # Let the model read all the tool results before deciding what to do next.
    response = llm_with_tools.invoke(messages_to_send)

messages_to_send.append(response)
print(response.content)


We have Cold Coffee and Chocolate Cold Coffee. Which one would you like?


### Connecting to the SQL?

In [32]:
import sqlite3
from pathlib import Path
from uuid import uuid4
from difflib import get_close_matches
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.tools import tool

db_path = "kiosk.sqlite3"

def connect_db():
    db = sqlite3.connect(db_path)
    db.row_factory = sqlite3.Row
    db.execute("PRAGMA foreign_keys = ON")
    return db

# The search_menu cell below loads its ID/name mapping beside the tool.


### IF customer exists get customer's user_id else create customer and get user_id

In [33]:
def find_or_create_customer(name, email):
    email = email.strip().lower()
    with connect_db() as db:
        row = db.execute("SELECT customer_id FROM customers WHERE email = ?", (email,)).fetchone()
        if row:
            return row["customer_id"]
        customer_id = str(uuid4())
        db.execute("INSERT INTO customers (customer_id, name, email) VALUES (?, ?, ?)",
                   (customer_id, name, email))
        return customer_id

# Demo login: reuse this customer if the email already exists.
# The order tools below use this ID; the LLM never supplies it.
signed_in_customer_id = find_or_create_customer("Student", "student@example.com")
print("Signed-in customer ID:", signed_in_customer_id)

Signed-in customer ID: 970d81dd-7edf-4b34-9252-ae7fb6400910


### WHat Do you have

In [43]:
class GetMenuInput(BaseModel):
    pass

@tool(args_schema=GetMenuInput)
def get_menu():
    """Read active products with names, IDs, current prices in paise and stock."""
    # Read the latest prices and stock from SQLite.
    with connect_db() as db:
        rows = db.execute("SELECT product_id, name, price_paise, stock_quantity FROM products WHERE active = 1")
        # Convert SQLite rows into normal dictionaries for the LLM.
        return [dict(row) for row in rows]

# Example: {"cold coffee": "cold_coffee", ...}
name_to_id = {name.lower(): product_id for product_id, name in product_names.items()}


In [37]:
get_menu.invoke({})

[{'product_id': 'cold_coffee',
  'name': 'Cold Coffee',
  'price_paise': 12000,
  'stock_quantity': 19},
 {'product_id': 'paneer_tikka',
  'name': 'Paneer Tikka',
  'price_paise': 22000,
  'stock_quantity': 12},
 {'product_id': 'veg_sandwich',
  'name': 'Veg Sandwich',
  'price_paise': 10000,
  'stock_quantity': 15},
 {'product_id': 'lemonade',
  'name': 'Lemonade',
  'price_paise': 6000,
  'stock_quantity': 25}]

In [41]:
class SearchMenuInput(BaseModel):
    query: str = Field(min_length=1, description="Product name requested by the customer.")

@tool(args_schema=SearchMenuInput)
def search_menu(query: str):
    """Find products by name, including minor typos; return real IDs, prices and stock."""
    # 1. Match the customer's wording to menu names.
    matched_names = get_close_matches(query.lower(), list(name_to_id), n=3, cutoff=0.6)
    # 2. Fetch the matching products with current price and stock.
    results = []
    with connect_db() as db:
        for matched_name in matched_names:
            # Customer wording -> matched menu name -> actual product ID.
            product_id = name_to_id[matched_name]
            row = db.execute(
                "SELECT product_id, name, price_paise, stock_quantity FROM products WHERE product_id = ? AND active = 1",
                (product_id,),
            ).fetchone()
            if row:
                results.append(dict(row))
    return results

print("Name to ID mapping:", name_to_id)
search_menu.invoke({"query": "cold coffe"})

NameError: name 'name_to_id' is not defined

In [39]:
tools = [get_menu, search_menu]


from langchain_core.messages import HumanMessage, SystemMessage

# Only the search tool is available in this example.
tools = [get_menu, search_menu]
tool_repo = {tool.name: tool for tool in tools}
llm_with_tools = llm.bind_tools(tools)

system_message = SystemMessage(content="""You are a helpful kiosk assistant.
Use search_menu to find products before discussing matching items.
Use product names and IDs from the tool result; never invent them.
If multiple products match, ask which one the customer wants.
If no products match, explain that and ask for another product name.
Do not place an order. Give one clear answer after receiving the results.
Do not print IDs of products or anything
""")

user_query = input("What would you like to look for? ")
messages_to_send = [system_message, HumanMessage(content=user_query)]

response = llm_with_tools.invoke(messages_to_send)
while response.tool_calls:
    # Preserve the AI message containing the requested tools.
    messages_to_send.append(response)
    for tool_call in response.tool_calls:
        tool_name = tool_call["name"]
        # A full tool call produces a ToolMessage linked to that call.
        tool_output = tool_repo[tool_name].invoke(tool_call)
        messages_to_send.append(tool_output)
    # Let the model read all the tool results before deciding what to do next.
    response = llm_with_tools.invoke(messages_to_send)

messages_to_send.append(response)
print(response.content)


We have Cold Coffee, Paneer Tikka, Veg Sandwich, and Lemonade.


In [40]:
messages_to_send

[SystemMessage(content='You are a helpful kiosk assistant.\nUse search_menu to find products before discussing matching items.\nUse product names and IDs from the tool result; never invent them.\nIf multiple products match, ask which one the customer wants.\nIf no products match, explain that and ask for another product name.\nDo not place an order. Give one clear answer after receiving the results.\nDo not print IDs of products or anything\n', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What all is available?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'tool_get_menu_OAn2pMH2VHrVIsgI2a2F', 'function': {'arguments': '{}', 'name': 'get_menu'}, 'type': 'function', 'index': 0}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 130, 'total_tokens': 140, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens'